In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
df = pd.read_csv('/Reviews.csv')
df = df[['Text']]
df.dropna(inplace=True)
df = df.iloc[:10000].copy()
df.head()

,Text
0,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...
2,This is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...


In [11]:
stop_words = set(stopwords.words('english'))

In [12]:
def preprocess(text):
    # a. Lowercase
    text = text.lower()

    # b. Remove punctuation and special characters
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

    # c. Tokenize
    tokens = word_tokenize(text)

    # d. Remove stopwords
    filtered = [word for word in tokens if word not in stop_words and word.isalpha()]

    # e. Join back
    return " ".join(filtered)


In [13]:
df['cleaned'] = df['Text'].apply(preprocess)
df.head()

,Text,cleaned
0,I have bought several of the Vitality canned d...,bought several vitality canned dog food produc...
1,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled jumbo salted peanuts p...
2,This is a confection that has been around a fe...,confection around centuries light pillowy citr...
3,If you are looking for the secret ingredient i...,looking secret ingredient robitussin believe f...
4,Great taffy at a great price. There was a wid...,great taffy great price wide assortment yummy ...


In [14]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['cleaned'])


In [15]:
def get_similar_reviews(query, top_k=5):
    # a. Preprocess the query
    cleaned_query = preprocess(query)

    # b. Convert to vector
    query_vec = vectorizer.transform([cleaned_query])

    # c. Compute cosine similarity
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # d. Get top k indices
    top_indices = similarities.argsort()[-top_k:][::-1]

    # e. Return original and cleaned text
    results = df.iloc[top_indices][['Text', 'cleaned']]
    return results


In [16]:
print("🔍 Query: 'great product with fast shipping'")
print(get_similar_reviews("great product with fast shipping"))

print("\n🔍 Query: 'disappointed'")
print(get_similar_reviews("disappointed"))


🔍 Query: 'great product with fast shipping'
                                                   Text  \
5226  Enjoyed the product and they also provided ver...   
8021  The tea is good and fresh. We enjoy it. The sh...   
5057  These are very good, they were a great price a...   
7073  My daughter lives in Hawaii and sent me some g...   
6034  The energy drink is a great product. The shipp...   

                                                cleaned  
5226  enjoyed product also provided fast shipping ne...  
8021  tea good fresh enjoy shipping fast cost reason...  
5057  good great price fast free shipping purchase e...  
7073  daughter lives hawaii sent great coffee keurig...  
6034  energy drink great product shipping price craz...  

🔍 Query: 'disappointed'
                                                   Text  \
3151  I am a bit disappointed.  The flavor was not w...   
4378  The product is very good. Way too expensive an...   
788   I was disappointed in this product because I 